# Workflow for Neosurf-on-Neosurf MaSIF search

All-vs-all MaSIF-Neosurf search: many ligand-bound **target** complexes are queried
against a large library of ligand-bound **seed** complexes. The pipeline has four stages,
each fanned out as a SLURM array job and each reading its inputs from disk, so **any stage
can be re-run on its own** after running the config cell.

1. **Prepare input** – download + trim + repair target/seed complexes.
2. **MaSIF preprocess** – surface descriptors for every complex.
3. **MaSIF search** – the all-vs-all neosurf-on-neosurf search.
4. **Gather + enrich metrics** – assemble result tables and derived metrics (what the explorer app reads).

> **Reproducibility:** SLURM submission is guarded by `SUBMIT_SLURM` (default `False`) so
> re-running a cell never launches jobs. The gather/enrich cells only read and write CSVs.
> Set `SUBMIT_SLURM = True` in the config cell only when you actually intend to submit.

In [ ]:
import os
import sys
import glob
import subprocess
import numpy as np
import pandas as pd

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

# ----- Source code on the python path -----
sys.path.insert(0, os.path.join(repo_root, "masif_seed_search/source"))
sys.path.insert(0, os.path.join(repo_root, "scripts/python"))
prepare_input_py = os.path.join(repo_root, "scripts/python/prepare_input.py")

# ----- Settings -----
N_ARRAY_JOBS = 500
EVOEF2_BIN = os.path.join(repo_root, "EvoEF2/EvoEF2")
N_SEED = None                 # limit seed list for testing (None = all)
SUBMIT_SLURM = False          # guard: cells submit jobs only when this is True
SEED_SITE_CUTOFF = 4.5        # A; seed patch-near-ligand cutoff (config.sh CUTOFF)

# ----- Directories -----
data_dir = os.path.join(repo_root, "data")
input_dir = os.path.join(data_dir, "input"); os.makedirs(input_dir, exist_ok=True)
seed_list_csv = os.path.join(data_dir, "human_reference_proteome_liganded_pdbs/human_reference_proteome_pdb_ligands_split.csv")

processing_dir = os.path.join(data_dir, "processing"); os.makedirs(processing_dir, exist_ok=True)
prep_input_proc_dir = os.path.join(processing_dir, "1_prep_input"); os.makedirs(prep_input_proc_dir, exist_ok=True)
masif_preprocess_proc_dir = os.path.join(processing_dir, "2_masif_preprocess"); os.makedirs(masif_preprocess_proc_dir, exist_ok=True)
masif_search_proc_dir = os.path.join(processing_dir, "3_masif_search"); os.makedirs(masif_search_proc_dir, exist_ok=True)
enrich_metrics_proc_dir = os.path.join(processing_dir, "4_enrich_metrics"); os.makedirs(enrich_metrics_proc_dir, exist_ok=True)

preprocess_dir = os.path.join(data_dir, "preprocess"); os.makedirs(preprocess_dir, exist_ok=True)
masif_search_out_dir = os.path.join(data_dir, "masif_search"); os.makedirs(masif_search_out_dir, exist_ok=True)
master_subset_dir = os.path.join(masif_search_out_dir, "subset"); os.makedirs(master_subset_dir, exist_ok=True)
query_targets_list = os.path.join(masif_search_out_dir, "query_targets.txt")

# Surfaces/pdbs used to (re)compute seed candidate patch counts in stage 4.
benchmark_surfaces_dir = os.path.join(preprocess_dir, "data_preparation", "01-benchmark_surfaces")
benchmark_pdbs_dir = os.path.join(preprocess_dir, "data_preparation", "01-benchmark_pdbs")

# ----- Key manifests / result tables (each stage reads these from disk) -----
preprocess_manifest_csv = os.path.join(masif_preprocess_proc_dir, "df_preprocess_manifest.csv")
preprocess_ok_csv = os.path.join(masif_preprocess_proc_dir, "df_preprocess_ok.csv")
results_csv = os.path.join(masif_search_proc_dir, "df_results_all.csv")
results_dedup_csv = os.path.join(masif_search_proc_dir, "df_results_dedup.csv")
enriched_all_csv = os.path.join(enrich_metrics_proc_dir, "df_results_all.csv")
enriched_dedup_csv = os.path.join(enrich_metrics_proc_dir, "df_results_dedup.csv")
n_target_sites_csv = os.path.join(enrich_metrics_proc_dir, "n_target_sites.csv")
n_seed_candidates_csv = os.path.join(enrich_metrics_proc_dir, "n_seed_candidates.csv")

## Stage 1 — Prepare input `.pdb` / `.sdf` files

For each complex: download structure + ligand from RCSB, trim to the target chain + one ligand residue, run EvoEF2 RepairStructure, and merge the ligand back. Fanned out as a SLURM array job.

In [ ]:
df_seed = pd.read_csv(seed_list_csv)

# Use only the first N_SEED rows for testing 
if N_SEED is not None:
    df_seed = df_seed.head(N_SEED)
    
print(f"df_seed.shape: {df_seed.shape}")
df_seed.head()

In [ ]:
# Prepare input files for a single complex
df_input_subset = df_seed[df_seed["pdb_id"] == "9CUO"]

df_input_subset.to_csv(os.path.join(data_dir, f"prepare_input.csv"), index=False)

cmd = [
    "python",
    prepare_input_py,
    "--input_csv", os.path.join(data_dir, f"prepare_input.csv"),
    "--outdir", input_dir,
    "--out_csv", os.path.join(data_dir, f"prepare_input_out.csv"),
    "--evoef2_bin", EVOEF2_BIN
]
print(cmd)
# subprocess.run(cmd)


In [ ]:
# Split the seed list into N_ARRAY_JOBS subsets and submit the prepare-input array job.
input_subset_dir = os.path.join(prep_input_proc_dir, "input_subsets"); os.makedirs(input_subset_dir, exist_ok=True)
output_subset_dir = os.path.join(prep_input_proc_dir, "output_subsets"); os.makedirs(output_subset_dir, exist_ok=True)

df_seed.to_csv(os.path.join(prep_input_proc_dir, "df_seed_input.csv"), index=False)
for i, chunk in enumerate(np.array_split(df_seed, N_ARRAY_JOBS)):
    chunk.to_csv(os.path.join(input_subset_dir, f"input_{i+1}.csv"), index=False)

cmd = [
    "sbatch", f"--array=1-{N_ARRAY_JOBS}", "scripts/slurm/prepare_input_array.sh",
    input_subset_dir, input_dir, output_subset_dir, EVOEF2_BIN,
]
print(" ".join(map(str, cmd)))
if SUBMIT_SLURM:
    subprocess.run(cmd, check=True)
else:
    print("SUBMIT_SLURM is False - not submitting (set it True in the config cell to submit).")

In [ ]:
# Gather all output .csv files into a single df_input_prepared
df_input_prepared = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_input_prepared = pd.concat([df_input_prepared, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_input_prepared.shape: {df_input_prepared.shape}")

# Split into success and failed rows
df_preprocess_manifest = df_input_prepared[
    df_input_prepared['pdb_path'].notna() & df_input_prepared['ligand_path'].notna()
]
df_input_failed = df_input_prepared[
    df_input_prepared['pdb_path'].isna() | df_input_prepared['ligand_path'].isna()
]
print(f"Successfully prepared {df_preprocess_manifest.shape[0]} complexes.")
print(f"Failed to prepare input files for {df_input_failed.shape[0]} complexes.")
print(f"Failed entries:")
df_input_failed.head()


## Stage 2 — MaSIF preprocessing

Computes molecular surfaces + learned descriptors for every prepared complex. Idempotent: rows whose descriptors already exist are skipped.

In [ ]:
# Preprocess subset directories (manifest/ok paths come from the config cell).
preprocess_input_subset_dir = os.path.join(masif_preprocess_proc_dir, "input_subsets"); os.makedirs(preprocess_input_subset_dir, exist_ok=True)
preprocess_output_subset_dir = os.path.join(masif_preprocess_proc_dir, "output_subsets"); os.makedirs(preprocess_output_subset_dir, exist_ok=True)

In [ ]:
# Round-robin split the manifest into subsets and submit the preprocess array job.
df_preprocess_manifest.to_csv(preprocess_manifest_csv, index=False)
for i in range(N_ARRAY_JOBS):
    df_preprocess_manifest.iloc[i::N_ARRAY_JOBS].to_csv(
        os.path.join(preprocess_input_subset_dir, f"input_{i+1}.csv"), index=False)

cmd = [
    "sbatch", f"--array=1-{N_ARRAY_JOBS}", "scripts/slurm/preprocess_array.sh",
    preprocess_input_subset_dir, preprocess_output_subset_dir,
]
print(" ".join(map(str, cmd)))
if SUBMIT_SLURM:
    subprocess.run(cmd, check=True)
else:
    print("SUBMIT_SLURM is False - not submitting (set it True in the config cell to submit).")

In [ ]:
# Gather preprocess output subsets
df_preprocess_results = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(preprocess_output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_preprocess_results = pd.concat([df_preprocess_results, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_preprocess_results.shape: {df_preprocess_results.shape}")

df_preprocess_ok = df_preprocess_results[
    df_preprocess_results["status"].isin(["success", "skipped"])
]
df_preprocess_ok.to_csv(preprocess_ok_csv, index=False)
df_preprocess_failed = df_preprocess_results[df_preprocess_results["status"] == "error"]

print(f"Preprocessed successfully or skipped: {df_preprocess_ok.shape[0]}")
print(f"Preprocess errors: {df_preprocess_failed.shape[0]}")
print("Failed entries:")
df_preprocess_failed.head()

## Stage 3 — MaSIF neosurf-on-neosurf search

The all-vs-all step: each query target is searched against the full seed library, anchored automatically on each side's largest ligand (`--target-auto-neosurf` / `--seed-auto-neosurf`).

In [ ]:
# seed_id is the canonical pipeline identifier computed by prepare_input.
# Use the manifest target column directly instead of re-deriving from seed fields.
assert df_preprocess_ok["target"].notna().all(), "Some successful rows are missing target"
df_preprocess_ok["seed_id"] = df_preprocess_ok["target"]
df_preprocessed_seeds = df_preprocess_ok[df_preprocess_ok["split"] == "seed"]            # seed-ligand complexes
df_preprocessed_targets = df_preprocess_ok[df_preprocess_ok["split"] == "target"]        # known E3 ligase-ligand complexes
seed_ids = df_preprocessed_seeds["seed_id"].tolist()
print("Number of seed ids: ", len(seed_ids))
seed_ids[0:5]

In [ ]:
# Deduplicate target complexes to unique ligase-compound complexes, keeping the
# row with the highest resolution (lowest numeric value).
#
# EXCEPTION: some critical ligands must NOT be deduplicated. Every PDB entry that
# carries these ligand codes is kept as a query target, because the single
# highest-resolution entry the dedup would otherwise pick is not always the
# optimal complex for the neosurf search.
ligand_code_to_include_all = [
    "EF2",  # thalidomide
    "Y70",  # pomalidomide
    "LVY",  # lenalidomide
    "3JF",  # VH032
    "4YY",  # VH101
    "6Z3",  # VH298
]

print(f"df_seed_targets.shape: {df_preprocessed_targets.shape}")
df_preprocessed_targets_sorted = df_preprocessed_targets.sort_values(by="resolution", ascending=True)

# Standard dedup for every non-critical ligand (one entry per uniprot_id + ligand_code) ...
df_dedup_standard = df_preprocessed_targets_sorted[
    ~df_preprocessed_targets_sorted["ligand_code"].isin(ligand_code_to_include_all)
].drop_duplicates(subset=["uniprot_id", "ligand_code"], keep="first")

# ... plus EVERY entry for the critical ligand codes (no deduplication).
df_include_all = df_preprocessed_targets_sorted[
    df_preprocessed_targets_sorted["ligand_code"].isin(ligand_code_to_include_all)
]

df_preprocessed_targets_dedup = pd.concat(
    [df_dedup_standard, df_include_all], ignore_index=True
).drop_duplicates(subset=["seed_id"], keep="first")

print(f"df_seed_targets_dedup.shape: {df_preprocessed_targets_dedup.shape}")
print(f"  standard-dedup targets:             {df_dedup_standard.shape[0]}")
print(f"  critical-ligand targets (all kept): {df_include_all.shape[0]}")
df_preprocessed_targets_dedup.head()

In [ ]:
# Function to submit a search job (array over seed subsets, looping every query target per task).
def submit_neosurf_search(query_targets, seed_ids, n_array_jobs, masif_search_out_dir, dry_run=True):
    """Write query_targets.txt + seed subset files and submit search_array.sh.

    Args:
        query_targets: list of query target ids (or a single id).
        seed_ids: seed ids to split into ``n_array_jobs`` subsets.
        n_array_jobs: number of array tasks / seed subsets.
        masif_search_out_dir: output + intermediate directory.
        dry_run: if True, only print the sbatch command (write nothing, submit nothing).
    """
    master_subset_dir = os.path.join(masif_search_out_dir, "subset")
    os.makedirs(master_subset_dir, exist_ok=True)
    query_target_txt = os.path.join(masif_search_out_dir, "query_targets.txt")
    if not isinstance(query_targets, (list, np.ndarray)):
        query_targets = [query_targets]

    if not dry_run:
        with open(query_target_txt, "w") as f:
            for target in query_targets:
                f.write(f"{target}\n")
        for idx, chunk in enumerate(np.array_split(seed_ids, n_array_jobs)):
            with open(os.path.join(master_subset_dir, f"{idx+1}"), "w") as sf:
                for seed in chunk:
                    sf.write(f"{seed}\n")

    cmd = [
        "sbatch", f"--array=1-{n_array_jobs}", "scripts/slurm/search_array.sh",
        query_target_txt, masif_search_out_dir, master_subset_dir,
    ]
    print(" ".join(map(str, cmd)))
    if not dry_run:
        subprocess.run(cmd, check=True)

In [ ]:
# Submit the search for every query target (the critical-ligand-inclusive dedup set from above).
query_targets = df_preprocessed_targets_dedup["seed_id"].tolist()
print(f"query targets: {len(query_targets)} | seeds: {len(seed_ids)}")
submit_neosurf_search(query_targets, seed_ids, N_ARRAY_JOBS, masif_search_out_dir, dry_run=not SUBMIT_SLURM)

## Stage 4 — Gather results & enrich metrics

Assemble the per-site `clustered_matches` csvs into `df_results_all.csv`, merge metadata and
deduplicate to `df_results_dedup.csv` (both under `3_masif_search/`), then compute enriched
metrics into `4_enrich_metrics/` — the tables the explorer app reads. These cells only read
and write CSVs, so they are safe to re-run without submitting any jobs.

In [ ]:
# Gather every clustered_matches csv into df_results_all.
csv_files = glob.glob(os.path.join(masif_search_out_dir, "*", "clustered_matches", "*.csv"))
df_results = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Loaded {len(df_results)} rows from {df_results['target'].nunique()} targets ({len(csv_files)} files).")
df_results.to_csv(results_csv, index=False)
df_results.head()

In [ ]:
# Merge preprocess metadata, deduplicate to one row per (target, matched_protein, cluster_id),
# and add n_match_ligands.
df_results = pd.read_csv(results_csv)
df_preprocess_ok = pd.read_csv(preprocess_ok_csv)

target_meta = df_preprocess_ok[["target", "gene_name"]].rename(columns={"gene_name": "target_gene_name"})
seed_meta = df_preprocess_ok[["target", "uniprot_id", "gene_name", "recommendedName", "mw"]].rename(columns={
    "target": "matched_protein", "uniprot_id": "matched_uniprot_id", "gene_name": "matched_gene_name",
    "recommendedName": "matched_recommendedName", "mw": "matched_mw"})
df_results = df_results.merge(target_meta, on="target", how="left").merge(seed_meta, on="matched_protein", how="left")

df_results_dedup = (df_results.sort_values("score", ascending=False)
                    .drop_duplicates(subset=["target", "matched_protein", "cluster_id"]))

# n_match_ligands: number of unique (target, matched_protein) pairs per (target_gene_name, matched_gene_name).
combo = (df_results_dedup.groupby(["target_gene_name", "matched_gene_name"])
         .apply(lambda x: x.drop_duplicates(subset=["target", "matched_protein"]).shape[0])
         .reset_index(name="n_match_ligands"))
df_results_dedup = df_results_dedup.merge(combo, on=["target_gene_name", "matched_gene_name"], how="left")

df_results_dedup.to_csv(results_dedup_csv, index=False)
print(f"df_results_dedup: {len(df_results_dedup)} rows / {df_results_dedup['target'].nunique()} targets")
df_results_dedup.head()

### Enrich metrics

Adds the following per-row metrics (all computed from committed inputs — no container/SLURM):

| column | definition |
|---|---|
| `target_mw` | molecular weight of the target complex |
| `n_target_sites` | number of anchor sites on the target (`site_*` dirs under `data/masif_search/<target>/`) |
| `n_seed_candidates` | number of seed surface patches near the seed ligand (vertices within `SEED_SITE_CUTOFF` Å of the seed's largest HET residue) |
| `total_n_patches` | `n_target_sites * n_seed_candidates` — search space size for that target×seed pair |
| `cluster_size_patch_normalized` | `cluster_size / total_n_patches` |
| `cluster_size_mw_normalized` | `cluster_size / (target_mw + matched_mw)` |

`n_seed_candidates` is cached per seed in `n_seed_candidates.csv`; newly seen seeds are computed from the surface geometry and appended.

In [ ]:
# Enrichment helpers (pure-python; no container needed).
from Bio.PDB import PDBParser
_ENRICH_PARSER = PDBParser(QUIET=True)


def _read_ply_xyz(path):
    """Return the (N, 3) vertex coordinates of a MaSIF surface .ply (ascii or binary)."""
    with open(path, "rb") as f:
        head = b""
        while b"end_header" not in head:
            head += f.readline()
        text = head.decode("latin1")
        nv = int([l for l in text.splitlines() if l.startswith("element vertex")][0].split()[2])
        fmt = [l for l in text.splitlines() if l.startswith("format")][0].split()[1]
        if fmt == "ascii":
            return np.array([[float(x) for x in f.readline().split()[:3]] for _ in range(nv)])
        tm = {"float": "f4", "double": "f8", "uchar": "u1", "int": "i4", "uint": "u4",
              "char": "i1", "short": "i2", "ushort": "u2"}
        vprops, in_vertex = [], False
        for l in text.splitlines():
            if l.startswith("element vertex"):
                in_vertex = True; continue
            if l.startswith("element") and "vertex" not in l:
                in_vertex = False
            if l.startswith("property") and in_vertex:
                vprops.append(l.split())
        dt = np.dtype([(p[-1], "<" + tm[p[1]]) for p in vprops])
        arr = np.frombuffer(f.read(dt.itemsize * nv), dtype=dt, count=nv)
        return np.stack([arr["x"], arr["y"], arr["z"]], 1)


def _ligand_anchor_coords(pdb_path):
    """Heavy-atom coords of the largest HET residue (matches masif find_ligand_anchor)."""
    st = _ENRICH_PARSER.get_structure("s", pdb_path)
    groups = {}
    for res in st.get_residues():
        if not res.id[0].strip():
            continue
        groups.setdefault((res.parent.id, res.id[1], res.get_resname().strip()), []).append(res)
    best, best_count = None, -1
    for residues in groups.values():
        c = sum(1 for r in residues for a in r if a.element != "H")
        if c > best_count:
            best_count, best = c, residues
    if best is None:
        return None
    return np.array([a.coord for r in best for a in r if a.element != "H"])


def compute_n_seed_candidates(seed_id):
    """# seed surface vertices within SEED_SITE_CUTOFF of the seed's ligand anchor."""
    ply = os.path.join(benchmark_surfaces_dir, f"{seed_id}.ply")
    pdb = os.path.join(benchmark_pdbs_dir, f"{seed_id}.pdb")
    if not (os.path.exists(ply) and os.path.exists(pdb)):
        return np.nan
    anchor = _ligand_anchor_coords(pdb)
    if anchor is None or len(anchor) == 0:
        return np.nan
    V = _read_ply_xyz(ply)
    dmin = np.sqrt(((V[:, None, :] - anchor[None, :, :]) ** 2).sum(-1)).min(1)
    return int((dmin < SEED_SITE_CUTOFF).sum())

In [ ]:
# Compute enriched metrics for both tables and write to 4_enrich_metrics/.
df_preprocess_ok = pd.read_csv(preprocess_ok_csv)
mw = df_preprocess_ok.set_index("target")["mw"]

df_all = pd.read_csv(results_csv)       # raw gathered results
df_dedup = pd.read_csv(results_dedup_csv)  # metadata-merged + deduplicated

# n_target_sites: number of anchor sites per target (site_* dirs).
targets = pd.unique(pd.concat([df_all["target"], df_dedup["target"]]))
n_target_sites = {t: len(glob.glob(os.path.join(masif_search_out_dir, t, "site_*"))) for t in targets}
pd.Series(n_target_sites, name="n_target_sites").to_csv(n_target_sites_csv)

# n_seed_candidates: per seed, cached; compute + append any newly-seen seeds.
if os.path.exists(n_seed_candidates_csv):
    nsc = pd.read_csv(n_seed_candidates_csv, index_col=0)["n_seed_candidates"].to_dict()
else:
    nsc = {}
seeds = pd.unique(pd.concat([df_all["matched_protein"], df_dedup["matched_protein"]]))
new_seeds = [s for s in seeds if s not in nsc]
for s in new_seeds:
    nsc[s] = compute_n_seed_candidates(s)
if new_seeds:
    pd.Series(nsc, name="n_seed_candidates").sort_index().to_csv(n_seed_candidates_csv)
print(f"n_target_sites for {len(n_target_sites)} targets; computed {len(new_seeds)} new seed candidate counts")


def add_enrichment(df, is_dedup):
    df = df.copy()
    orig = list(df.columns)
    if "matched_mw" not in df.columns:
        df["matched_mw"] = df["matched_protein"].map(mw)
    df["target_mw"] = df["target"].map(mw)
    df["n_target_sites"] = df["target"].map(n_target_sites)
    df["n_seed_candidates"] = df["matched_protein"].map(nsc)
    df["total_n_patches"] = df["n_target_sites"] * df["n_seed_candidates"]
    df["cluster_size_patch_normalized"] = df["cluster_size"] / df["total_n_patches"]
    df["cluster_size_mw_normalized"] = df["cluster_size"] / (df["target_mw"] + df["matched_mw"])
    metrics = ["n_target_sites", "n_seed_candidates", "total_n_patches",
               "cluster_size_patch_normalized", "cluster_size_mw_normalized"]
    tail = ["target_mw"] + metrics if is_dedup else ["target_mw", "matched_mw"] + metrics
    return df[orig + tail]


add_enrichment(df_all, is_dedup=False).to_csv(enriched_all_csv, index=False)
df_enriched_dedup = add_enrichment(df_dedup, is_dedup=True)
df_enriched_dedup.to_csv(enriched_dedup_csv, index=False)
print(f"wrote enriched all + dedup ({len(df_enriched_dedup)} rows) to {enrich_metrics_proc_dir}")
df_enriched_dedup.head()